# 🎤 Comparador Vocal Multi-Artista

## Extensión del Vocal Pitch Analysis Pipeline — Idea C

**Integrantes:** Balda Javier · Caracoix Juan · Casas Facundo  
**Institución:** Universidad Católica Argentina  
**Materia:** Análisis y Procesamiento de Datos Streaming  

---

## 📌 Descripción

Construimos un **dataset comparativo de 6 perfiles vocales** usando el motor de síntesis
de audio del challenge original (`generate_demo.py`) extendido con nuevos perfiles.
Para cada artista extraemos 16 métricas que cubren cuatro dimensiones:

| Dimensión | Métricas |
|-----------|----------|
| **Rango vocal** | nota_grave/aguda, Hz extremos, rango en semitonos y octavas |
| **Perfil de registro** | % del tiempo en grave/medio-grave/medio-agudo/agudo/sobreagudo |
| **Dinámica** | intensidad media, variación, % de alta intensidad |
| **Estilo melódico** | notas distintas, densidad de cambios, ratio agudo/grave |

### Perfiles incluidos

| ID | Perfil | Rango real | Referencia estilística |
|----|--------|------------|------------------------|
| `bogdan_style` | Tenor Dramático | A2–C6 | Bogdan Shuvalov — S.O.S |
| `baritono` | Barítono Lírico | F2–A4 | Estilo Sinatra / Barrientos |
| `soprano` | Soprano Lírica | C4–F6 | Estilo Callas / Netrebko |
| `tenor_pop` | Tenor Pop / Belting | B2–E5 | Estilo Freddie / Adam Lambert |
| `contratenor` | Contratenor | G3–C6 | Estilo Jaroussky / Scholl |
| `mezzo` | Mezzo-Soprano | F3–A5 | Estilo Bartoli / DiDonato |

### Integración con el pipeline original

```
artist_profiles.py          vocal_comparador.py         dashboard HTML
───────────────────         ────────────────────         ──────────────
ARTISTS dict               extract_metrics_from_        comparador_
  ├─ VOCAL_SEQUENCE  ──►   sequence(profile)      ──►  dashboard.html
  ├─ timbre params          ├─ 16 métricas                (Chart.js)
  └─ generate_audio()       └─ timeline por nota
                                   │
pipeline.py (existente)            │ load_from_csv()
realtime_frames.csv ──────────────►┘
(artista real → integrable)
```

## Sección 1 · Instalación y setup

In [ ]:
!pip install numpy scipy matplotlib pandas scikit-learn -q
print('✅ Dependencias instaladas.')

# Si subiste los archivos a Colab:
# from google.colab import files
# files.upload()  # subir artist_profiles.py y vocal_comparador.py

In [ ]:
# Importar módulos del proyecto
from artist_profiles import ARTISTS, generate_all, note_freq, generate_audio
from vocal_comparador import (
    build_dataset, save_dataset,
    load_from_csv, extract_metrics_from_sequence,
    classify_register
)

import json
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

print(f'✅ Módulos cargados.')
print(f'   Perfiles disponibles: {list(ARTISTS.keys())}')

## Sección 2 · Generación de audios sintéticos (opcional)

In [ ]:
from scipy.io import wavfile

# Listar perfiles con sus rangos
print('Perfiles vocales disponibles:\n')
print(f'  {"ID":<20} {"Nombre":<38} {"Rango":<15} {"Octavas":>8}')
print('  ' + '─'*85)
for aid, p in ARTISTS.items():
    seq = p['sequence']
    lo  = min(note_freq(n) for n,*_ in seq)
    hi  = max(note_freq(n) for n,*_ in seq)
    n_lo = min(seq, key=lambda x: note_freq(x[0]))[0]
    n_hi = max(seq, key=lambda x: note_freq(x[0]))[0]
    oct_ = 12 * math.log2(hi/lo) / 12
    print(f'  {aid:<20} {p["name"]:<38} {n_lo}–{n_hi:<11} {oct_:>7.2f} oct')

In [ ]:
# Generar todos los audios WAV (descomentá para ejecutar)
# Los archivos se guardan en data/artists/

# paths = generate_all(out_dir='data/artists', sr=22050)
# print(f'\n✅ Audios generados: {paths}')

print('ℹ️  Generación de audio omitida (no necesaria para las métricas).')
print('   Descomentá la celda para generar los WAV si los necesitás para el análisis con pYIN.')

## Sección 3 · Construcción del dataset de métricas

In [ ]:
import os

# ── Construir dataset sintético de los 6 perfiles ────────────────
print('[BUILD] Calculando métricas de los 6 perfiles vocales...')
dataset = build_dataset()
print(f'\n✅ Dataset construido: {len(dataset)} artistas')

# ── Integrar artista real (si tenés realtime_frames.csv) ─────────
CSV_REAL = 'results/realtime_frames.csv'  # o 'realtime_frames.csv'
if os.path.exists(CSV_REAL):
    bogdan_real = load_from_csv(
        CSV_REAL,
        artist_id='bogdan_real',
        name='Bogdan (análisis real)',
        tipo='Tenor dramático',
        genero='Pop/Rock dramático',
        color='#f87171'
    )
    dataset.append(bogdan_real)
    print(f'  + Bogdan (CSV real): {bogdan_real["nota_grave"]}–{bogdan_real["nota_aguda"]} / {bogdan_real["rango_octavas"]} oct')

# Convertir a DataFrame
skip_cols = {'timeline', 'ref'}
df = pd.DataFrame([{k:v for k,v in d.items() if k not in skip_cols} for d in dataset])
df = df.set_index('id')
print(f'\nDataFrame: {df.shape[0]} artistas × {df.shape[1]} columnas')
display(df[['name','tipo','nota_grave','nota_aguda','rango_octavas','hz_medio','nota_modal','n_notas_distintas']].round(2))

In [ ]:
# Guardar dataset
Path('results').mkdir(exist_ok=True)
paths = save_dataset(dataset, out_dir='results')

print('\nColumnas del CSV:')
df_csv = pd.read_csv('results/artist_metrics.csv')
print(df_csv.dtypes)

## Sección 4 · Análisis exploratorio — rango y registro

In [ ]:
# ── Figura 1: Rango vocal — diagrama de barras de frecuencia ─────
fig, ax = plt.subplots(figsize=(13, 5))
fig.suptitle('Comparación de rangos vocales por artista', fontsize=13, fontweight='bold')

COLORS = {d['id']: d['color'] for d in dataset}
names  = [d['name'].split('(')[0].strip() for d in dataset]
ids    = [d['id'] for d in dataset]

y_pos = range(len(dataset))
for i, d in enumerate(dataset):
    # Barra de rango
    ax.barh(i, d['hz_aguda'] - d['hz_grave'],
            left=d['hz_grave'],
            height=0.55,
            color=d['color'],
            alpha=0.75,
            edgecolor=d['color'])
    # Marcador nota grave
    ax.scatter(d['hz_grave'], i, s=60, color=d['color'], zorder=5)
    # Marcador nota aguda
    ax.scatter(d['hz_aguda'], i, s=60, color=d['color'], marker='D', zorder=5)
    # Marcador Hz medio
    ax.axvline(x=d['hz_medio'], ymin=(i-0.25)/len(dataset),
               ymax=(i+0.25)/len(dataset), color='white', linewidth=1.5, alpha=0.8)
    # Etiquetas
    ax.text(d['hz_grave'] - 15, i, d['nota_grave'],
            va='center', ha='right', fontsize=9, color=d['color'])
    ax.text(d['hz_aguda'] + 15, i, d['nota_aguda'],
            va='center', ha='left', fontsize=9, color=d['color'])

ax.set_yticks(list(y_pos))
ax.set_yticklabels(names, fontsize=10)
ax.set_xlabel('Frecuencia (Hz)', fontsize=11)
ax.set_xscale('log')  # escala log para mejor visualización
ax.set_xticks([65, 130, 262, 523, 1047, 2093])
ax.set_xticklabels(['C2\n65', 'C3\n130', 'C4\n262', 'C5\n523', 'C6\n1047', 'C7\n2093'])
ax.grid(True, alpha=0.2, axis='x')
ax.set_facecolor('#0f172a')
fig.patch.set_facecolor('#0f172a')
ax.tick_params(colors='#94a3b8')
ax.xaxis.label.set_color('#94a3b8')

# Leyenda
handles = [
    mpatches.Patch(color='white', alpha=0.8, label='Hz medio (línea vertical)'),
    plt.scatter([],[], s=60, c='white', marker='o', label='Nota más grave'),
    plt.scatter([],[], s=60, c='white', marker='D', label='Nota más aguda'),
]
ax.legend(handles=handles, loc='lower right', fontsize=9, facecolor='#1e293b', labelcolor='#94a3b8')

plt.tight_layout()
plt.savefig('results/rango_vocal.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()
print('✅ Guardado: results/rango_vocal.png')

In [ ]:
# ── Figura 2: Distribución de registros — stacked bar ────────────
REG_KEYS   = ['pct_grave','pct_medio_grave','pct_medio_agudo','pct_agudo','pct_sobreagudo']
REG_LABELS = ['Grave','Medio-grave','Medio-agudo','Agudo','Sobreagudo']
REG_COLORS = ['#6366f1','#8b5cf6','#38bdf8','#34d399','#f472b6']

fig, ax = plt.subplots(figsize=(13, 5))
fig.suptitle('Distribución de registros vocales (% del tiempo)', fontsize=13, fontweight='bold')

x    = np.arange(len(dataset))
left = np.zeros(len(dataset))

for key, label, color in zip(REG_KEYS, REG_LABELS, REG_COLORS):
    vals = [d[key] for d in dataset]
    bars = ax.barh(x, vals, left=left, height=0.6,
                   label=label, color=color, alpha=0.85, edgecolor='none')
    # Etiqueta si el segmento es suficientemente ancho
    for i, (v, l) in enumerate(zip(vals, left)):
        if v > 8:
            ax.text(l + v/2, i, f'{v:.0f}%',
                    va='center', ha='center', fontsize=8.5,
                    color='white', fontweight='bold')
    left += vals

ax.set_yticks(x)
ax.set_yticklabels(names, fontsize=10)
ax.set_xlabel('% del tiempo con voz', fontsize=11)
ax.set_xlim(0, 105)
ax.legend(loc='lower right', fontsize=9, facecolor='#1e293b', labelcolor='#e2e8f0')
ax.grid(True, alpha=0.2, axis='x')
ax.set_facecolor('#0f172a')
fig.patch.set_facecolor('#0f172a')
ax.tick_params(colors='#94a3b8')
ax.xaxis.label.set_color('#94a3b8')

plt.tight_layout()
plt.savefig('results/distribucion_registros.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()
print('✅ Guardado: results/distribucion_registros.png')

## Sección 5 · Análisis comparativo — métricas de estilo

In [ ]:
# ── Figura 3: Radar chart — perfil vocal multidimensional ────────
from matplotlib.patches import FancyArrowPatch
import matplotlib.pyplot as plt
import numpy as np

categories = [
    'Rango\n(oct)', 'Hz medio\n(norm)', 'Densidad\nmelódica',
    '% Agudo+\nSobreagudo', 'Intens.\nmedia', 'Notas\ndistintas'
]
N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # cerrar el polígono

def normalize(val, lo, hi):
    return max(0, min(1, (val - lo) / (hi - lo)))

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
fig.patch.set_facecolor('#0f172a')
ax.set_facecolor('#0f172a')
fig.suptitle('Perfil vocal multidimensional', fontsize=13, fontweight='bold', color='#f1f5f9', y=0.98)

for d in dataset:
    vals = [
        normalize(d['rango_octavas'],       1.5, 4.0),
        normalize(d['hz_medio'],            100, 900),
        normalize(d['densidad_melodica'],   0.2, 0.6),
        normalize((d['pct_agudo']+d['pct_sobreagudo']), 0, 50),
        normalize(d['intensidad_media'],    0.4, 1.0),
        normalize(d['n_notas_distintas'],   10,  30),
    ]
    vals += vals[:1]
    label = d['name'].split('(')[0].strip().split(' ')[:2]
    label = ' '.join(label)

    ax.plot(angles, vals, color=d['color'], linewidth=1.8, label=label)
    ax.fill(angles, vals, color=d['color'], alpha=0.08)
    # Marcadores en vértices
    ax.scatter(angles[:-1], vals[:-1], s=30, color=d['color'], zorder=5)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=10, color='#94a3b8')
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['25%','50%','75%','100%'], size=8, color='#475569')
ax.grid(color='#1e3a5f', linewidth=0.5)
ax.spines['polar'].set_color('#1e3a5f')

ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1),
          fontsize=9.5, facecolor='#1e293b', labelcolor='#e2e8f0',
          framealpha=0.9)

plt.tight_layout()
plt.savefig('results/radar_perfil_vocal.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()
print('✅ Guardado: results/radar_perfil_vocal.png')

In [ ]:
# ── Figura 4: Scatter Hz medio vs Rango (burbuja = densidad melódica) ──
fig, ax = plt.subplots(figsize=(10, 6))
fig.suptitle('Hz medio vs Rango vocal · tamaño = densidad melódica', fontsize=12, fontweight='bold')

for d in dataset:
    size = d['densidad_melodica'] * 1200
    ax.scatter(d['hz_medio'], d['rango_octavas'],
               s=size, color=d['color'], alpha=0.6, edgecolors=d['color'], linewidth=1.5)
    name_short = d['name'].split('(')[0].strip()
    ax.annotate(name_short,
                xy=(d['hz_medio'], d['rango_octavas']),
                xytext=(10, 5), textcoords='offset points',
                fontsize=9, color=d['color'])

ax.set_xlabel('Frecuencia media (Hz)', fontsize=11)
ax.set_ylabel('Rango vocal (octavas)', fontsize=11)
ax.grid(True, alpha=0.2)
ax.set_facecolor('#0f172a')
fig.patch.set_facecolor('#0f172a')
ax.tick_params(colors='#94a3b8')
for spine in ax.spines.values(): spine.set_color('#1e3a5f')

plt.tight_layout()
plt.savefig('results/scatter_hz_rango.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()
print('✅ Guardado: results/scatter_hz_rango.png')

## Sección 6 · Clustering — agrupación automática de perfiles

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Features para clustering
FEATURES = [
    'rango_octavas', 'hz_medio', 'densidad_melodica',
    'pct_grave', 'pct_medio_grave', 'pct_medio_agudo',
    'pct_agudo', 'pct_sobreagudo',
    'intensidad_media', 'ratio_agudo_grave',
]

X = df[FEATURES].values.astype(float)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Silhouette score para k=2,3,4
print('Silhouette scores (k-means):')
for k in range(2, 5):
    if k <= len(dataset):
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X_scaled)
        score = silhouette_score(X_scaled, labels) if len(set(labels)) > 1 else 0
        print(f'  k={k}: {score:.3f}')

# Clustering jerárquico con k=3
hc = AgglomerativeClustering(n_clusters=3, linkage='ward')
clusters = hc.fit_predict(X_scaled)

df['cluster'] = clusters
print('\nClusters asignados:')
for i, (idx, row) in enumerate(df.iterrows()):
    print(f'  Cluster {row["cluster"]}: {row["name"]}')

In [ ]:
# ── PCA 2D + clusters ─────────────────────────────────────────────
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(9, 6))
fig.suptitle('Clustering de perfiles vocales (PCA 2D)', fontsize=12, fontweight='bold')
fig.patch.set_facecolor('#0f172a')
ax.set_facecolor('#0f172a')

cluster_colors = ['#38bdf8','#f472b6','#34d399']
for i, d in enumerate(dataset):
    color_dot = [d['color'] for d in dataset][i]
    cc = cluster_colors[clusters[i]]
    ax.scatter(X_pca[i, 0], X_pca[i, 1], s=220,
               c=color_dot, alpha=0.85,
               edgecolors=cc, linewidth=2.5, zorder=5)
    name_s = [d['name'] for d in dataset][i].split('(')[0].strip()
    ax.annotate(name_s, xy=(X_pca[i,0], X_pca[i,1]),
                xytext=(8, 6), textcoords='offset points',
                fontsize=9, color=color_dot)

# Círculos de cluster
for c in range(3):
    pts = X_pca[clusters == c]
    if len(pts) > 0:
        cx, cy = pts.mean(axis=0)
        r = max(0.5, np.max(np.sqrt((pts[:,0]-cx)**2 + (pts[:,1]-cy)**2)) + 0.3)
        circle = plt.Circle((cx, cy), r, fill=False,
                             color=cluster_colors[c], linewidth=1,
                             linestyle='--', alpha=0.4)
        ax.add_patch(circle)
        ax.text(cx, cy - r - 0.15, f'Cluster {c+1}',
                ha='center', fontsize=9, color=cluster_colors[c])

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.0f}% varianza)', color='#94a3b8')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.0f}% varianza)', color='#94a3b8')
ax.tick_params(colors='#94a3b8')
ax.grid(True, alpha=0.15)
for spine in ax.spines.values(): spine.set_color('#1e3a5f')

plt.tight_layout()
plt.savefig('results/pca_clustering.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()
print('✅ Guardado: results/pca_clustering.png')
print(f'\nVarianza explicada: {sum(pca.explained_variance_ratio_)*100:.0f}% en 2 componentes')

## Sección 7 · Integrar artista real desde pipeline.py

In [ ]:
import os

# Si tenés el CSV del challenge (realtime_frames.csv), cargarlo aquí:
CSV_PATH = 'results/realtime_frames.csv'  # ajustar ruta

if os.path.exists(CSV_PATH):
    m_real = load_from_csv(
        CSV_PATH,
        artist_id='bogdan_real',
        name='Bogdan (análisis pYIN real)',
        tipo='Tenor dramático',
        genero='Pop/Rock',
        color='#f87171'
    )
    print('✅ Artista real cargado:')
    print(f'   Rango:   {m_real["nota_grave"]} – {m_real["nota_aguda"]} ({m_real["rango_octavas"]} oct)')
    print(f'   Hz medio: {m_real["hz_medio"]} Hz')
    print(f'   Notas distintas: {m_real["n_notas_distintas"]}')

    # Comparación directa: sintético vs real
    m_synth = extract_metrics_from_sequence(ARTISTS['bogdan_style'])
    print('\nComparación sintético vs análisis pYIN real:')
    for k in ['rango_octavas','hz_medio','n_notas_distintas','densidad_melodica','ratio_agudo_grave']:
        diff = m_real[k] - m_synth[k]
        arrow = '↑' if diff > 0 else '↓'
        print(f'  {k:<30} sintético={m_synth[k]:.3f}  real={m_real[k]:.3f}  '
              f'diff={arrow}{abs(diff):.3f}')
else:
    print(f'ℹ️  {CSV_PATH} no encontrado.')
    print('   Ejecutá pipeline.py del challenge y copiá realtime_frames.csv aquí.')

## Sección 8 · Exportar y dashboard

In [ ]:
# Tabla resumen final
resumen = df[['name','tipo','rango_octavas','hz_medio','nota_modal',
              'densidad_melodica','pct_alta_intensidad','ratio_agudo_grave']].copy()
resumen.columns = ['Artista','Tipo','Rango (oct)','Hz medio','Nota modal',
                   'Dens. melód.','% alta intens.','Ratio agudo/grave']
print('=== TABLA COMPARATIVA FINAL ===')
display(resumen.round(3))

In [ ]:
# Abrir dashboard en Colab
import os
DASHBOARD_PATH = 'results/comparador_dashboard.html'
if os.path.exists(DASHBOARD_PATH):
    from IPython.display import IFrame
    display(IFrame(src=DASHBOARD_PATH, width='100%', height='1000'))
else:
    print('Copiá comparador_dashboard.html a results/ y abrí en navegador.')

In [ ]:
# Descargar archivos (en Colab)
archivos = [
    'results/artist_dataset.json',
    'results/artist_metrics.csv',
    'results/rango_vocal.png',
    'results/distribucion_registros.png',
    'results/radar_perfil_vocal.png',
    'results/pca_clustering.png',
]
try:
    from google.colab import files
    for f in archivos:
        if os.path.exists(f):
            files.download(f)
    print('✅ Archivos descargados.')
except Exception:
    print('ℹ️  Archivos disponibles en results/:')
    for f in archivos:
        if os.path.exists(f):
            print(f'  ✓ {f}')

---

## 📋 Conclusiones

### Hallazgos principales

| Hallazgo | Evidencia |
|----------|-----------|
| Bogdan tiene el mayor rango (3.25 oct) | Único perfil con graves extremos A2 Y sobreagudos C6 |
| El barítono domina los graves (43% en rango grave) | Hz medio más bajo: 212 Hz |
| La soprano tiene el Hz medio más alto (692 Hz) | 100% del tiempo en medio-agudo o superior |
| El contratenor supera a la soprano en nota aguda | Falsete hasta C6 (masculino) |
| Bogdan es el más denso melódicamente (0.44 cam/s) | 40 cambios de nota en 90 segundos |

### Clustering
Los 6 artistas se agrupan naturalmente en **3 clusters**:
- **Cluster 1 (graves):** Barítono, Tenor Pop → Hz medio bajo, predominio registro grave/medio-grave
- **Cluster 2 (medios):** Bogdan-style, Mezzo → amplio rango, alta densidad melódica
- **Cluster 3 (agudos):** Soprano, Contratenor → Hz medio alto, predominio registros agudos

### Extensiones posibles
- Agregar más artistas reales via `load_from_csv()` + pipeline.py
- Entrenar un clasificador que prediga el tipo vocal a partir de las métricas (Idea A)
- Añadir análisis de anomalías por artista (Idea D) para comparar "densidad de eventos"